# 06 — Test d'idempotence Silver Iceberg

**Objectif** : vérifier qu'ingérer deux fois le même fichier NDJSON
ne crée pas de doublons dans la table Iceberg  
**Mécanisme** : `tray_id = SHA-256(machine_id | candled_at_utc)`  
**Test** : rejouer les 5 premiers fichiers NDJSON et vérifier
que le nombre de lignes et les comptages restent identiques

In [1]:
import os 
import json 
import base64
import hashlib
import re 
import warnings
from pathlib import Path
import numpy as np 
import pandas as pd
import pyarrow as pa 
from pyiceberg.catalog.sql import SqlCatalog
from pyiceberg.expressions import And, EqualTo
from dotenv import load_dotenv

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)

load_dotenv()
print("Ok")


Ok


In [2]:
import sys
sys.path.append("../src")
from catalog import get_catalog, ADLS_URI
from schema import get_or_create_silver_trays

catalog = get_catalog()
table   = get_or_create_silver_trays(catalog, ADLS_URI)

Table 'silver.trays' chargée — 180 snapshots.
  Location : abfss://silver@dlsecatcandlingfrcedev.dfs.core.windows.net/trays_iceberg
  Format   : Iceberg v2
  Champs   : 27


In [4]:
# Etat Initial avant rejeu 

MACHINE_ID = "PMAF-C012501"

df_before = table.scan(
    row_filter=And(
        EqualTo("machine_id", MACHINE_ID),
        EqualTo("year",  2026),
        EqualTo("month", 5),
        EqualTo("day",   15),
    )
).to_pandas()

n_before       = len(df_before)
snapshots_before = len(table.snapshots())
tray_ids_before  = set(df_before["tray_id"].tolist())

print("État AVANT rejeu")
print(f"Lignes totales     : {n_before}")
print(f"Snapshots          : {snapshots_before}")
print(f"tray_ids uniques   : {len(tray_ids_before)}")
print(f"Doublons tray_id   : {df_before['tray_id'].duplicated().sum()}")

État AVANT rejeu
Lignes totales     : 1344
Snapshots          : 180
tray_ids uniques   : 1344
Doublons tray_id   : 0


In [5]:
RE_MATRIX = re.compile(r"\[(\d+)\]\[(\d+)\]$")

def decode_message(raw_line):
    envelope = json.loads(raw_line)
    body     = json.loads(base64.b64decode(envelope["Body"]).decode("utf-8"))
    return {
        "enqueued_at": envelope["EnqueuedTimeUtc"],
        "device_id":   envelope["SystemProperties"]["connectionDeviceId"],
        "machine_id":  body["process_serial_number"],
        "box":         body["box"],
        "values": {v["id"].split(".")[-1]: v["value"] for v in body["values"]},
        "timestamps": {v["id"].split(".")[-1]: v["timestamp"] for v in body["values"]},
    }

def extract_matrix(values, tag_prefix):
    matrix = [[0] * 10 for _ in range(15)]
    for tag_name, val in values.items():
        if not tag_name.startswith(tag_prefix):
            continue
        m = RE_MATRIX.search(tag_name)
        if not m:
            continue
        row, col = int(m.group(1)) - 1, int(m.group(2)) - 1
        if 0 <= row < 15 and 0 <= col < 10:
            matrix[row][col] = int(val) if val is not None else 0
    return matrix

def matrix_to_compact(matrix):
    return "".join(str(cell) for row in matrix for cell in row)

def compute_counts(matrix):
    flat = [cell for row in matrix for cell in row]
    return {
        "n_total": len(flat), "n_fertile": flat.count(1),
        "n_clear": flat.count(3), "n_early_dead": flat.count(2),
        "n_late_dead": flat.count(4), "n_missing": flat.count(0),
    }

def compute_tray_id(machine_id, candled_at_utc):
    return hashlib.sha256(f"{machine_id}|{candled_at_utc}".encode()).hexdigest()

def parse_ndjson_file(filepath):
    rows = []
    with open(filepath, "r", encoding="utf-8") as f:
        lines = [l.strip() for l in f if l.strip()]
    for line in lines:
        msg = decode_message(line)
        if msg["values"].get("pmaf_trig") != "Tray":
            continue
        v              = msg["values"]
        candled_at_str = msg["timestamps"].get("pmaf_trig", msg["enqueued_at"])
        candled_at     = pd.to_datetime(candled_at_str, utc=True)
        matrix         = extract_matrix(v, "final_candled_eggs")
        counts         = compute_counts(matrix)
        light_mat      = extract_matrix(v, "laser1_light_transmitted_eggs")
        rows.append({
            "tray_id":                  compute_tray_id(msg["machine_id"], candled_at_str),
            "machine_id":               msg["machine_id"],
            "candled_at":               candled_at,
            "candled_date":             candled_at.date(),
            "flock":                    int(v.get("flock_number", 0)),
            "trolley":                  str(v.get("trolley_name", "")),
            "tray_seq":                 int(v.get("setter_tray_number_flock", 0)),
            "flock_name":               str(v.get("flock_name", "")),
            "trolley_name":             str(v.get("trolley_name", "")),
            "caliber":                  str(v.get("caliber", "")),
            "setter_tray_number_flock": int(v.get("setter_tray_number_flock", 0)),
            **counts,
            "is_count_consistent":      counts["n_total"] == 150,
            "matrix_compact":           matrix_to_compact(matrix),
            "light_flat":               [cell for row in light_mat for cell in row],
            "alarm_emergency_stop":     int(v.get("alarm_emergency_stop", 0)),
            "alarm_air_pressure_fault": int(v.get("alarm_air_pressure_fault", 0)),
            "processed_at":             pd.Timestamp.utcnow(),
            "bronze_path":              filepath.name,
            "year":                     candled_at.year,
            "month":                    candled_at.month,
            "day":                      candled_at.day,
        })
    return pd.DataFrame(rows)

def df_to_arrow(df):
    def clean(val):
        if val is None:
            return None
        return [0 if (v is None or (isinstance(v, float) and np.isnan(v)))
                else int(v) for v in val]
    df = df.copy()
    df["light_flat"]               = df["light_flat"].apply(clean)
    df["candled_at"]               = pd.to_datetime(df["candled_at"], utc=True)
    df["processed_at"]             = pd.to_datetime(df["processed_at"], utc=True)
    df["candled_date"]             = pd.to_datetime(df["candled_date"]).dt.date
    for col in ["flock", "tray_seq", "setter_tray_number_flock", "n_total",
                "n_fertile", "n_clear", "n_early_dead", "n_late_dead",
                "n_missing", "alarm_emergency_stop", "alarm_air_pressure_fault"]:
        df[col] = df[col].fillna(0).astype("int32")
    df["is_count_consistent"] = df["is_count_consistent"].astype("bool")
    df["trolley"]             = df["trolley"].astype("str")
    df["year"]                     = df["year"].astype("int32")
    df["month"]                    = df["month"].astype("int32")
    df["day"]                      = df["day"].astype("int32")

    return pa.Table.from_pandas(df, schema=pa.schema([
        pa.field("tray_id",                  pa.string(),               nullable=False),
        pa.field("machine_id",               pa.string(),               nullable=False),
        pa.field("candled_at",               pa.timestamp("us", "UTC"), nullable=False),
        pa.field("candled_date",             pa.date32(),               nullable=False),
        pa.field("flock",                    pa.int32(),                nullable=False),
        pa.field("trolley",                  pa.string(),               nullable=False),
        pa.field("tray_seq",                 pa.int32(),                nullable=False),
        pa.field("flock_name",               pa.string(),               nullable=True),
        pa.field("trolley_name",             pa.string(),               nullable=True),
        pa.field("caliber",                  pa.string(),               nullable=True),
        pa.field("setter_tray_number_flock", pa.int32(),                nullable=True),
        pa.field("n_total",                  pa.int32(),                nullable=False),
        pa.field("n_fertile",                pa.int32(),                nullable=False),
        pa.field("n_clear",                  pa.int32(),                nullable=False),
        pa.field("n_early_dead",             pa.int32(),                nullable=False),
        pa.field("n_late_dead",              pa.int32(),                nullable=False),
        pa.field("n_missing",                pa.int32(),                nullable=False),
        pa.field("is_count_consistent",      pa.bool_(),                nullable=False),
        pa.field("matrix_compact",           pa.string(),               nullable=False),
        pa.field("light_flat",               pa.list_(pa.int32()),      nullable=True),
        pa.field("alarm_emergency_stop",     pa.int32(),                nullable=True),
        pa.field("alarm_air_pressure_fault", pa.int32(),                nullable=True),
        pa.field("processed_at",             pa.timestamp("us", "UTC"), nullable=False),
        pa.field("bronze_path",              pa.string(),               nullable=True),
        pa.field("year",                     pa.int32(),                nullable=False),
        pa.field("month",                    pa.int32(),                nullable=False),
        pa.field("day",                      pa.int32(),                nullable=False),

    ]), preserve_index=False)

print("Fonctions définies.")

Fonctions définies.


In [6]:
NDJSON_DIR = Path("../data/ndjson_sim")
files      = sorted(NDJSON_DIR.glob("**/*.json"))[:5]

print(f"Fichiers à rejouer : {[f.name for f in files]}\n")

# Collecter les tray_ids qu'on va rejouer
tray_ids_replayed = set()

for filepath in files:
    df_batch = parse_ndjson_file(filepath)
    if df_batch.empty:
        continue

    tray_ids_replayed.update(df_batch["tray_id"].tolist())

    # Filtrer les tray_ids déjà présents dans Iceberg
    already_exists = tray_ids_replayed & tray_ids_before
    new_trays      = df_batch[~df_batch["tray_id"].isin(tray_ids_before)]

    print(f"{filepath.name} — {len(df_batch)} plateaux — "
          f"{len(already_exists)} déjà existants — "
          f"{len(new_trays)} nouveaux")

    if new_trays.empty:
        print(f"  → Tous les tray_ids existent déjà, skip.")
        continue

    # N'écrire que les nouveaux
    table.append(df_to_arrow(new_trays))
    print(f"  → {len(new_trays)} lignes écrites.")

print("\nRejeu terminé.")

Fichiers à rejouer : ['04_32_0.json', '04_33_0.json', '04_34_0.json', '04_35_0.json', '04_36_0.json']

04_32_0.json — 2 plateaux — 2 déjà existants — 0 nouveaux
  → Tous les tray_ids existent déjà, skip.
04_33_0.json — 11 plateaux — 13 déjà existants — 0 nouveaux
  → Tous les tray_ids existent déjà, skip.
04_34_0.json — 9 plateaux — 22 déjà existants — 0 nouveaux
  → Tous les tray_ids existent déjà, skip.
04_35_0.json — 6 plateaux — 28 déjà existants — 0 nouveaux
  → Tous les tray_ids existent déjà, skip.
04_36_0.json — 4 plateaux — 32 déjà existants — 0 nouveaux
  → Tous les tray_ids existent déjà, skip.

Rejeu terminé.


In [7]:
df_after = table.scan(
    row_filter=And(
        EqualTo("machine_id",   MACHINE_ID),
        EqualTo("candled_date", "2026-05-15"),
    )
).to_pandas()

n_after          = len(df_after)
snapshots_after  = len(table.snapshots())
tray_ids_after   = set(df_after["tray_id"].tolist())
doublons         = df_after["tray_id"].duplicated().sum()

print("État APRÈS rejeu") 
print(f"Lignes totales     : {n_after}")
print(f"Snapshots          : {snapshots_after}")
print(f"tray_ids uniques   : {len(tray_ids_after)}")
print(f"Doublons tray_id   : {doublons}")

État APRÈS rejeu
Lignes totales     : 1344
Snapshots          : 180
tray_ids uniques   : 1344
Doublons tray_id   : 0


In [8]:
print("Verdict idempotence\n")

lignes_ok    = n_after == n_before
doublons_ok  = doublons == 0
tray_ids_ok  = tray_ids_after == tray_ids_before

print(f"Lignes inchangées  : {'OK' if lignes_ok   else 'KO'} ({n_before} → {n_after})")
print(f"Zéro doublon       : {'OK' if doublons_ok  else 'KO'} ({doublons} doublons)")
print(f"tray_ids identiques: {'OK' if tray_ids_ok  else 'KO'}")

if lignes_ok and doublons_ok and tray_ids_ok:
    print("\n Idempotence validée — le pipeline est safe pour rejouer des fichiers.")
else:
    print("\n Problème détecté — le pipeline n'est pas idempotent.")

Verdict idempotence

Lignes inchangées  : OK (1344 → 1344)
Zéro doublon       : OK (0 doublons)
tray_ids identiques: OK

 Idempotence validée — le pipeline est safe pour rejouer des fichiers.
